# IALS (Implicit Alternating Least Squares )

В этом ноутбуке я построю IALS модель, протестирую её и сохраню 

## Импорт библиотек 

In [37]:
import pandas as pd
import numpy as np
import scipy 
import os
import implicit 
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from implicit.evaluation import train_test_split as implicit_split 
from implicit.evaluation import precision_at_k, mean_average_precision_at_k, ranking_metrics_at_k
import implicit.gpu
import optuna
import optuna.visualization as vis
import plotly


In [38]:
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

## Загрузка данных

#### Выгружаем матрицу

In [39]:
results_path = Path("../../../results/matrices")
user_item_sparse = scipy.sparse.load_npz(results_path/'artnail_user_item_sparse.npz')

#### Выгружаем очищенные таблицы

In [40]:
clean_path = Path("../../../Tables/CleanTable")
users_clean = pd.read_csv(clean_path / 'users_clean_final.csv')
items_clean = pd.read_csv(clean_path / 'items_clean_final.csv')

In [41]:
f'Размеры матрицы: {user_item_sparse.shape}'

'Размеры матрицы: (2893, 258)'

## IALS model 

#### Train/Validation split для метрик

In [42]:
train_sparse, val_sparse = implicit_split(user_item_sparse, train_percentage=0.9, random_state=42)

In [43]:
print(f"  Train: {train_sparse.nnz} ({train_sparse.nnz/user_item_sparse.nnz:.1%})")
print(f"  Val:   {val_sparse.nnz} ({val_sparse.nnz/user_item_sparse.nnz:.1%})")

  Train: 877 (90.4%)
  Val:   93 (9.6%)


#### Построение базовой модели 

In [44]:
IALS_model = model = implicit.als.AlternatingLeastSquares(
    factors=64,           # Размер эмбеддингов (пользователи × услуги)
    regularization=0.1,   # L2 регуляризация (0.01-0.5)
    iterations=50,        # ALS итераций (сходимость)
    random_state=42,      # Репродуцируемость
    use_gpu=False,        # CPU 
    num_threads=4         # Потоки (4-8 оптимально)
)

Обучение 

In [45]:
CONFIDENCE_SCALE = 15
IALS_model.fit(train_sparse*CONFIDENCE_SCALE)

print(f"  User embeddings: {model.user_factors.shape}")
print(f"  Item embeddings: {model.item_factors.shape}")

  0%|          | 0/50 [00:00<?, ?it/s]

  User embeddings: (2893, 64)
  Item embeddings: (258, 64)


#### Оценка качества базовой модели 

IALS будет выполнять функции подбора кандидатов, а функцию ранжирования будет выполнять `CutBoost` модель, поэтому в оценке `IALS` главной метрикой буду считать Recall@K, где K будет большим.

Цель не отобрать конкретные 10 лучших вариантов, а отобрать 50 кандидатов, главное чтобы среди них были хорошие кандидаты и CutBoost увидел их

In [46]:
metrics = ranking_metrics_at_k(IALS_model, train_sparse, val_sparse, K=50)
metrics

  0%|          | 0/91 [00:00<?, ?it/s]

{'precision': 0.5806451612903226,
 'map': 0.029722000906388105,
 'ndcg': 0.12825269952781929,
 'auc': 0.6866756822032526}

Пояснение: В implicit версии 0.7.2 метрика Precision@K считает по факту как Recall@K, поэтому мы будем интарпретировать precision как recall. 

In [47]:
recall_50 = metrics['precision']
f'Recall@50 = {recall_50:.6f}'

'Recall@50 = 0.580645'

__Из 50 кандидатов 57% релевантны__. Это хороший показатель для CutBoost. 

Также  `AUC` = 0.6819 показывает, что модель уверенно отличает положительные и негативные оценки.

Из остальных метрик можно понять, что модель плохо ранжирует отобранных кандидатов, но для нашей ситуации это не страшно, т. к. ранжировать кандидатов будет CutBoost.

#### Random Search с  помощью Optuna

In [ ]:
def objective(trial):
    """Optuna objective: максимизируем Recall@50"""
    
    # Предлагаем параметры
    params = {
        # factors: 64 ±20  (48, 64, 80, 96)
        'factors': trial.suggest_int('factors', 48, 96, step=16),  
        
        # regularization: 0.1 ±0.08 (0.02-0.18) 
        'regularization': trial.suggest_float('regularization', 0.02, 0.18, log=False),
        
        # alpha: 1.0 ±0.8 (0.2-1.8) 
        'alpha': trial.suggest_float('alpha', 0.2, 1.8),
        
        # iterations: 50 ±15 (35-65) — сходимость ALS
        'iterations': trial.suggest_int('iterations', 35, 65),
        
        # Фиксированные 
        'use_gpu': False,
        'num_threads': 4,
        'random_state': 42
    }

    # Модель
    model = implicit.als.AlternatingLeastSquares(**params)
    model.fit(train_sparse.T.tocsr())

    # Метрики
    metrics = ranking_metrics_at_k(
        model, 
        train_user_items= train_sparse.T.tocsr(), 
        test_user_items = val_sparse.T.tocsr(), 
        K=50
    )

    recall50 = metrics['precision']

    # Callback для pruning (Optuna может прервать плохой trial)
    trial.report(recall50, step=1)
    if trial.should_prune():
        raise optuna.TrialPruned()

    print(f"✅ Trial {trial.number}: Recall@50={recall50:.4f} | {params}")
    return recall50


In [64]:
study = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner()
)

[I 2026-03-08 00:38:29,859] A new study created in memory with name: no-name-4d7c0965-f3b5-4d12-a963-530db2662980


In [62]:
study.optimize(objective, n_trials=20)
print(f"Лучший Recall@50: {study.best_value:.4f}")

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

[I 2026-03-08 00:36:44,306] Trial 0 finished with value: 0.07526881720430108 and parameters: {'factors': 128, 'regularization': 0.24420854259858868, 'alpha': 2.586676739010885, 'iterations': 30}. Best is trial 0 with value: 0.07526881720430108.


✅ Trial 0: Recall@50=0.0753 | {'factors': 128, 'regularization': 0.24420854259858868, 'alpha': 2.586676739010885, 'iterations': 30, 'random_state': 9, 'use_gpu': False}


  0%|          | 0/28 [00:00<?, ?it/s]

...

[I 2026-03-08 00:36:59,270] Trial 18 pruned. 


  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

[I 2026-03-08 00:37:00,570] Trial 19 pruned. 


Лучший Recall@50: 0.0860
